# Cell Type Annotation and Phenotyping

This notebook performs cell type identification and annotation using:
- Marker gene expression
- Automated annotation tools
- Manual curation
- scVI-tools for probabilistic modeling

**Input:** Preprocessed AnnData object from 01_preprocessing.ipynb

**Output:** Annotated AnnData with cell type labels

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=80, facecolor='white', frameon=False)

print(f"Scanpy version: {sc.__version__}")
print(f"scVI version: {scvi.__version__}")

## 1. Load Preprocessed Data

In [ ]:
# Define paths
DATA_DIR = Path("../data/processed")
OUTPUT_DIR = Path("../data/processed")
FIGURES_DIR = Path("../figures/02_phenotyping")

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Sample name
SAMPLE_NAME = "xenium_sample_01"

# Load preprocessed data
adata = sc.read_h5ad(DATA_DIR / f"{SAMPLE_NAME}_preprocessed.h5ad")

print(f"Loaded data shape: {adata.shape}")
print(f"Available clusters: {adata.obs['leiden_r0.5'].nunique()}")

## 2. Find Cluster Markers

In [ ]:
# Find marker genes for each cluster
sc.tl.rank_genes_groups(
    adata,
    groupby='leiden_r0.5',
    method='wilcoxon',
    key_added='rank_genes_leiden'
)

# Plot top marker genes
sc.pl.rank_genes_groups(adata, n_genes=20, sharey=False, key='rank_genes_leiden')
plt.savefig(FIGURES_DIR / 'marker_genes_by_cluster.png', dpi=300, bbox_inches='tight')
plt.show()

print("Marker gene analysis completed")

In [ ]:
# Get top marker genes as DataFrame
result = adata.uns['rank_genes_leiden']
groups = result['names'].dtype.names

marker_df = pd.DataFrame({
    group + '_' + key: result[key][group]
    for group in groups for key in ['names', 'scores']
})

# Save marker genes
marker_df.to_csv(OUTPUT_DIR / f"{SAMPLE_NAME}_marker_genes.csv")
print(f"Marker genes saved to {OUTPUT_DIR / f'{SAMPLE_NAME}_marker_genes.csv'}")

# Display top 5 markers per cluster
print("\nTop 5 marker genes per cluster:")
for cluster in groups[:5]:  # Show first 5 clusters
    print(f"\nCluster {cluster}:")
    print(marker_df[f"{cluster}_names"].head().tolist())

## 3. Define Cell Type Markers

Define known marker genes for major cell types (customize based on tissue type)

In [ ]:
# Define marker genes for common cell types
# Customize these based on your tissue type and expected cell populations

marker_genes = {
    'T cells': ['CD3D', 'CD3E', 'CD3G', 'CD8A', 'CD4'],
    'B cells': ['CD19', 'CD79A', 'CD79B', 'MS4A1'],
    'NK cells': ['NKG7', 'GNLY', 'NCAM1'],
    'Myeloid': ['CD14', 'CD68', 'LYZ', 'CSF1R'],
    'Macrophages': ['CD68', 'CD163', 'MSR1'],
    'Dendritic cells': ['CD1C', 'FCER1A', 'CLEC9A'],
    'Neutrophils': ['FCGR3B', 'CSF3R', 'S100A8', 'S100A9'],
    'Epithelial': ['EPCAM', 'KRT8', 'KRT18', 'KRT19'],
    'Endothelial': ['PECAM1', 'VWF', 'CDH5', 'ENG'],
    'Fibroblasts': ['COL1A1', 'COL1A2', 'DCN', 'LUM'],
    'Smooth muscle': ['ACTA2', 'MYH11', 'TAGLN'],
    'Proliferating': ['MKI67', 'TOP2A', 'PCNA'],
}

# Filter markers present in dataset
available_markers = {}
for cell_type, genes in marker_genes.items():
    present_genes = [g for g in genes if g in adata.var_names]
    if present_genes:
        available_markers[cell_type] = present_genes

print("Available marker genes by cell type:")
for ct, genes in available_markers.items():
    print(f"  {ct}: {', '.join(genes)}")

## 4. Visualize Marker Gene Expression

In [ ]:
# Create dotplot of marker genes
all_markers = [gene for genes in available_markers.values() for gene in genes]
all_markers = list(set(all_markers))  # Remove duplicates

if all_markers:
    sc.pl.dotplot(
        adata,
        all_markers[:30],  # Limit to 30 genes for visibility
        groupby='leiden_r0.5',
        dendrogram=True,
        standard_scale='var'
    )
    plt.savefig(FIGURES_DIR / 'marker_dotplot.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Create heatmap
    sc.pl.heatmap(
        adata,
        all_markers[:30],
        groupby='leiden_r0.5',
        swap_axes=True,
        standard_scale='var',
        cmap='RdBu_r'
    )
    plt.savefig(FIGURES_DIR / 'marker_heatmap.png', dpi=300, bbox_inches='tight')
    plt.show()

## 5. Score Cell Types

In [ ]:
# Score cells for each cell type based on marker expression
for cell_type, genes in available_markers.items():
    score_name = f'{cell_type}_score'
    sc.tl.score_genes(adata, genes, score_name=score_name)
    print(f"Scored {cell_type}")

# Plot cell type scores on UMAP
score_names = [f'{ct}_score' for ct in available_markers.keys()]

if score_names:
    n_plots = len(score_names)
    n_cols = 3
    n_rows = (n_plots + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows))
    axes = axes.flatten() if n_plots > 1 else [axes]
    
    for idx, score in enumerate(score_names[:n_plots]):
        sc.pl.umap(adata, color=score, ax=axes[idx], show=False, cmap='viridis')
    
    # Hide extra subplots
    for idx in range(n_plots, len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'celltype_scores_umap.png', dpi=300, bbox_inches='tight')
    plt.show()

## 6. Assign Cell Types Based on Scores

In [ ]:
# Assign cell types based on highest score
score_cols = [f'{ct}_score' for ct in available_markers.keys()]

if score_cols:
    # Get the cell type with maximum score for each cell
    scores_df = adata.obs[score_cols]
    adata.obs['predicted_celltype'] = scores_df.idxmax(axis=1).str.replace('_score', '')
    
    # Calculate confidence (difference between top 2 scores)
    top2_scores = np.sort(scores_df.values, axis=1)[:, -2:]
    adata.obs['celltype_confidence'] = top2_scores[:, 1] - top2_scores[:, 0]
    
    print("\nCell type distribution:")
    print(adata.obs['predicted_celltype'].value_counts())
    
    # Plot predicted cell types
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    sc.pl.umap(adata, color='predicted_celltype', ax=axes[0], show=False)
    sc.pl.umap(adata, color='celltype_confidence', ax=axes[1], show=False, cmap='viridis')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'predicted_celltypes.png', dpi=300, bbox_inches='tight')
    plt.show()

## 7. scVI-tools for Advanced Analysis (Optional)

Use scVI for probabilistic modeling and better integration

In [ ]:
# Setup anndata for scVI
scvi.model.SCVI.setup_anndata(
    adata,
    layer='counts',
    batch_key=None  # Add batch_key if you have batches
)

# Train scVI model
vae = scvi.model.SCVI(adata, n_layers=2, n_latent=30, gene_likelihood="nb")

print("Training scVI model...")
vae.train(max_epochs=200, early_stopping=True)

# Get latent representation
adata.obsm['X_scvi'] = vae.get_latent_representation()

# Compute neighbors and UMAP on scVI latent space
sc.pp.neighbors(adata, use_rep='X_scvi', key_added='scvi')
sc.tl.umap(adata, neighbors_key='scvi')
sc.tl.leiden(adata, resolution=0.5, neighbors_key='scvi', key_added='leiden_scvi')

print("scVI analysis completed")

In [ ]:
# Compare scVI clustering with marker-based prediction
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sc.pl.umap(adata, color='leiden_scvi', ax=axes[0], show=False, title='scVI Leiden')
sc.pl.umap(adata, color='predicted_celltype', ax=axes[1], show=False, title='Marker-based prediction')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'scvi_vs_markers.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Manual Refinement (if needed)

In [ ]:
# Create manual annotation dictionary based on cluster inspection
# This is a template - update based on your marker gene analysis

# Example manual annotation
cluster_annotation = {
    # Format: 'cluster_id': 'cell_type'
    # '0': 'T cells',
    # '1': 'B cells',
    # '2': 'Myeloid',
    # ... add more based on your analysis
}

# Apply manual annotation if provided
if cluster_annotation:
    adata.obs['manual_celltype'] = adata.obs['leiden_r0.5'].map(cluster_annotation)
    adata.obs['manual_celltype'] = adata.obs['manual_celltype'].fillna('Unknown')
    
    sc.pl.umap(adata, color='manual_celltype')
    plt.savefig(FIGURES_DIR / 'manual_annotation.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("No manual annotation provided. Using predicted cell types.")
    adata.obs['manual_celltype'] = adata.obs['predicted_celltype']

## 9. Final Cell Type Assignment

In [ ]:
# Use manual annotation as final if available, otherwise use predicted
adata.obs['celltype'] = adata.obs['manual_celltype']

print("\nFinal cell type distribution:")
print(adata.obs['celltype'].value_counts())

# Create comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 15))

sc.pl.umap(adata, color='celltype', ax=axes[0, 0], show=False, title='Cell Types')
sc.pl.umap(adata, color='leiden_r0.5', ax=axes[0, 1], show=False, title='Leiden Clusters')
sc.pl.umap(adata, color='total_counts', ax=axes[1, 0], show=False, cmap='viridis', title='Total Counts')
sc.pl.umap(adata, color='n_genes_by_counts', ax=axes[1, 1], show=False, cmap='viridis', title='N Genes')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'final_celltypes_overview.png', dpi=300, bbox_inches='tight')
plt.show()

## 10. Save Annotated Data

In [ ]:
# Save annotated data
output_file = OUTPUT_DIR / f"{SAMPLE_NAME}_annotated.h5ad"
adata.write_h5ad(output_file)

print(f"\nAnnotated data saved to: {output_file}")
print(f"Dataset shape: {adata.shape}")
print(f"\nCell type annotations:")
print(adata.obs['celltype'].value_counts())

# Export cell type assignments
celltype_export = adata.obs[['celltype', 'predicted_celltype', 'celltype_confidence']]
celltype_export.to_csv(OUTPUT_DIR / f"{SAMPLE_NAME}_celltype_assignments.csv")
print(f"\nCell type assignments exported to: {OUTPUT_DIR / f'{SAMPLE_NAME}_celltype_assignments.csv'}")

---

## Next Steps

The annotated data is ready for:
1. **Spatial analysis** (03_spatial_analysis.ipynb)
2. **Group-wise comparisons** (04_group_comparisons.ipynb)
3. **Tissue comparisons** (05_tissue_comparisons.ipynb)